# Создание датасета: `food` / `not_food` (v2)

Переписанная версия `create_dataset.ipynb`. Первая версия давала **100% на тесте** и при этом
**12 ошибок из 26** на обычных пользовательских запросах. Разбор причин:

| проблема v1 | следствие |
|---|---|
| 250 примеров, 125 сущностей на класс, каждая встречается один раз | задача сводится к «есть ли это слово в моём списке» |
| между классами всего 7 общих токенов, из них неоднозначен один (`паста`) | границу приходится проводить по словарю, а не по смыслу |
| все 250 примеров построены по шаблону `NP, adj, на столе.` | модель выучила шаблон: «Книга, толстая, на столе.» → `food` |
| нет ни одного примера на метафоры, отрицания, ложных друзей | переносные значения («рецепт успеха», «соль в оркестре») ломают модель |
| `train_test_split(.2)` без группировки + `eval == test` | 100% на тесте — артефакт сплита, а не качества |

**Что построено в этой версии**

* label policy, записанная явно, с решением всех краевых случаев;
* лексика сущностей (еда / не-еда) в формах `nom | gen | pl` + род;
* **разделяемые фреймы** — один и тот же синтаксический каркас для обоих классов, чтобы метку
  нельзя было вывести из формы предложения;
* хард-пакет: ложные друзья, метафоры, отрицания, съедобно-звучное несъедобное, бренды,
  жанровые сдвиги;
* surface-аугментации: регистр, пунктуация, опечатки, шум, keywords-only;
* **entity-level split** `train / validation / test / hard_test` — целые сущности не пересекаются;
* валидация датасета: утечки, баланс, keyword-baseline и TF-IDF baseline;
* публикация с `ClassLabel`, dataset card, опциональный `push_to_hub`.

Воспроизводимо: `SEED = 42`, генерация через `random.Random(SEED)`, порядок операций фиксирован.

**Порядок прогона.** Сначала посмотри на `REPORT` в конце — там asserts. Если какой-то assert
упал, чинить нужно его, а не обучение.

In [ ]:
import json
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42

DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "food_not_food_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# id2label фиксируем ЯВНО. В v1 он выводился из dataset['train'].unique('label')[::-1],
# то есть зависел от порядка строк и молча ломался при перезаливке датасета.
ID2LABEL = {0: "not_food", 1: "food"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

TEXT_COL = "caption"   # имя колонки прежнее: text_classification.ipynb продолжит работать

# сколько фреймов на одну сущность (и сколько surface-вариантов на один текст)
FRAMES_PER_ENTITY = 14
PERTURB_PROB = 0.45

HF_REPO_ID = "alexneo68/for_text_classification_food_not_food_v2"
PUSH_TO_HUB = True # True + `huggingface-cli login`, чтобы перезалить

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

## 1. Label policy

Метка — про **смысл**, а не про наличие слов. Это единственное решение, которое делает датасет
пригодным: без него разметка «на глаз» превращается в шум, и хард-пакет ниже просто
закрепил бы чьи-то догадки.

In [ ]:
LABEL_POLICY = {
    "version": "2.0",
    "positive": "food",
    "rule": (
        "1 — если текст относится к съедобным для человека продуктам или блюдам: прямое "
        "упоминание, запрос, описание, обсуждение, а также магазины/сервисы доставки еды "
        "и места питания. 0 — во всех остальных случаях."
    ),
    "sense_over_string": "важен смысл, а не наличие слов: «сливочное масло» — 1, «масляная картина» — 0",
    "negation": "отрицание не меняет метку: «я не ем сахар» — 1 (еда упомянута)",
    "out_of_scope": ["латиница", "code-switching", "английский текст"],
    "scope_note": (
        "датасет русскоязычный. Латиница, микс языков и английский текст вне распределения — "
        "ruBERT на них не обучен, и модель будет ошибаться; это не дефект разметки."
    ),
}

EDGES = [
    ("сливочное масло", 1, "масло в пищевом смысле"),
    ("масляная картина", 0, "масло в другом смысле"),
    ("соль с чесноком и перцем", 1, "пряность"),
    ("соль в оркестре", 0, "музыкальная соль"),
    ("я не ем сахар", 1, "отрицание не меняет метку"),
    ("в доме не было еды", 1, "то же"),
    ("столовая в офисе", 1, "место питания"),
    ("ВкусВилл", 1, "сеть продуктов"),
    ("Магнит", 0, "бытовая техника"),
    ("продукт смартфона", 0, "«продукт» в непищевом смысле"),
    ("корм для кошки", 0, "не для человека"),
    ("Apple представила процессор", 0, "другое значение слова"),
    ("яблоко от яблони", 0, "идиома"),
    ("Хрен с ним!", 0, "восклицание, а не овощ"),
    ("зубная паста", 0, "ложный друг"),
    ("Паста с морепродуктами", 1, "паста как еда"),
    ("Не будь мороженым", 0, "шутливое сравнение с человеком"),
    ("кость в парке", 0, "не для человека"),
    ("пробка от бутылки", 0, "съедобно звучит, но не еда"),
    ("микроволновка", 0, "на кухне, но не еда"),
]

POLICY_DF = pd.DataFrame(EDGES, columns=["текст", "метка_id", "обоснование"])
POLICY_DF["метка"] = POLICY_DF["метка_id"].map(ID2LABEL)
POLICY_DF = POLICY_DF[["текст", "метка", "обоснование"]]
print(LABEL_POLICY["rule"])
POLICY_DF

## 2. Лексика сущностей

Формат строки: `группа | метка | nom | gen | pl | род`.

* `группа` — это **сущность**, по ней делается сплит: все примеры одной сущности попадают в один
  split. Поэтому тест всегда проверяет незнакомые слова, а не знакомые.
* `gen` для родительного падежа («рецепт яблока»), `pl` для множественного, `род` — чтобы
  прилагательные согласовывались. Формат одинаковый для обоих классов.
* `группа` food и not_food с одинаковым корнем (`овощ:лук` и `hard:ff:лук:еда`) намеренно
  держатся раздельно: перенос значения — это отдельная задача, и её видно только на
  `hard_test`.

In [ ]:
FOOD_LEX = r"""
# --- фрукты и ягоды
food:фрукт:яблоко|food|яблоко|яблока|яблоки|с
food:фрукт:груша|food|груша|груши|груши|ж
food:фрукт:айва|food|айва|айвы|айвы|ж
food:фрукт:апельсин|food|апельсин|апельсина|апельсины|м
food:фрукт:мандарин|food|мандарин|мандарина|мандарины|м
food:фрукт:лимон|food|лимон|лимона|лимоны|м
food:фрукт:лайм|food|лайм|лайма|лаймы|м
food:фрукт:грейпфрут|food|грейпфрут|грейпфрута|грейпфруты|м
food:фрукт:киви|food|киви|киви|киви|с
food:фрукт:банан|food|банан|банана|бананы|м
food:фрукт:ананас|food|ананас|ананаса|ананасы|м
food:фрукт:манго|food|манго|манго|манго|с
food:фрукт:папайя|food|папайя|папайи|папайи|ж
food:фрукт:арбуз|food|арбуз|арбуза|арбузы|м
food:фрукт:дыня|food|дыня|дыни|дыни|ж
food:фрукт:персик|food|персик|персика|персики|м
food:фрукт:нектарин|food|нектарин|нектарина|нектарины|м
food:фрукт:абрикос|food|абрикос|абрикоса|абрикосы|м
food:фрукт:слива|food|слива|сливы|сливы|ж
food:фрукт:вишня|food|вишня|вишни|вишни|ж
food:фрукт:черешня|food|черешня|черешни|черешни|ж
food:фрукт:клубника|food|клубника|клубники|клубники|ж
food:фрукт:земляника|food|земляника|земляники|земляники|ж
food:фрукт:черника|food|черника|черники|черники|ж
food:фрукт:малина|food|малина|малины|малины|ж
food:фрукт:смородина|food|смородина|смородины|смородины|ж
food:фрукт:голубика|food|голубика|голубики|голубики|ж
food:фрукт:крыжовник|food|крыжовник|крыжовника|крыжовник|м
food:фрукт:гранат|food|гранат|граната|гранаты|м
food:фрукт:хурма|food|хурма|хурмы|хурмы|ж
food:фрукт:инжир|food|инжир|инжира|инжиры|м
food:фрукт:финики|food|финики|фиников|финики|м
food:фрукт:каштан|food|каштан|каштана|каштаны|м
# --- орехи и сухофрукты
food:орех:миндаль|food|миндаль|миндаля|миндали|м
food:орех:кешью|food|кешью|кешью|кешью|м
food:орех:фундук|food|фундук|фундука|фундуки|м
food:орех:арахис|food|арахис|арахиса|арахисы|м
food:орех:грецкий|food|грецкий орех|грецкого ореха|грецкие орехи|м
food:орех:кедровый|food|кедровый орех|кедрового ореха|кедровые орехи|м
food:сухофрукты:курага|food|курага|кураги|курага|ж
food:сухофрукты:чернослив|food|чернослив|чернослива|черносливы|м
food:сухофрукты:изюм|food|изюм|изюма|изюм|м
food:сухофрукты:сухофрукты|food|сухофрукты|сухофруктов|сухофрукты|м
# --- овощи и зелень
food:овощ:огурец|food|огурец|огурца|огурцы|м
food:овощ:помидор|food|помидор|помидора|помидоры|м
food:овощ:морковь|food|морковь|моркови|моркови|ж
food:овощ:картофель|food|картофель|картофеля|картофели|м
food:овощ:капуста|food|капуста|капусты|капусты|ж
food:овощ:брокколи|food|брокколи|брокколи|брокколи|ж
food:овощ:цветнаякапуста|food|цветная капуста|цветной капусты|цветные капусты|ж
food:овощ:брюссельскаякапуста|food|брюссельская капуста|брюссельской капусты|брюссельские капусты|ж
food:овощ:пекинскаякапуста|food|пекинская капуста|пекинской капусты|пекинские капусты|ж
food:овощ:пакчой|food|пак чой|пак чоя|пак чои|м
food:овощ:лук|food|лук|лука|луки|м
food:овощ:чеснок|food|чеснок|чеснока|чесноки|м
food:овощ:свекла|food|свёкла|свёклы|свёклы|ж
food:овощ:редис|food|редис|редиса|редисы|м
food:овощ:редька|food|редька|редьки|редьки|ж
food:овощ:дайкон|food|дайкон|дайкона|дайконы|м
food:овощ:шпинат|food|шпинат|шпината|шпинаты|м
food:овощ:салат|food|салат|салата|салаты|м
food:овощ:руккола|food|руккола|рукколы|рукколы|ж
food:овощ:базилик|food|базилик|базилика|базилики|м
food:овощ:петрушка|food|петрушка|петрушки|петрушки|ж
food:овощ:укроп|food|укроп|укропа|укропы|м
food:овощ:кинза|food|кинза|кинзы|кинзы|ж
food:овощ:перец|food|перец|перца|перцы|м
food:овощ:чили|food|чили|чили|чили|ж
food:овощ:баклажан|food|баклажан|баклажана|баклажаны|м
food:овощ:кабачок|food|кабачок|кабачка|кабачки|м
food:овощ:тыква|food|тыква|тыквы|тыквы|ж
food:овощ:спаржа|food|спаржа|спаржи|спаржи|ж
food:овощ:сельдерей|food|сельдерей|сельдерея|сельдереи|м
food:овощ:артишок|food|артишок|артишока|артишоки|м
food:овощ:кукуруза|food|кукуруза|кукурузы|кукурузы|ж
food:овощ:фасольстручковая|food|зелёная фасоль|зелёной фасоли|зелёные фасоли|ж
food:овощ:авокадо|food|авокадо|авокадо|авокадо|с
food:овощ:грибы|food|грибы|грибов|грибы|м
food:овощ:лисички|food|лисички|лисичек|лисички|ж
food:овощ:белыегрибы|food|белые грибы|белых грибов|белые грибы|м
# --- крупы, бобовые, макароны
food:крупы:горох|food|горох|гороха|горохи|м
food:крупы:фасоль|food|фасоль|фасоли|фасоль|ж
food:крупы:чечевица|food|чечевица|чечевицы|чечевицы|ж
food:крупы:нут|food|нут|нута|нуты|м
food:крупы:рис|food|рис|риса|рис|м
food:крупы:овсянка|food|овсянка|овсянки|овсянки|ж
food:крупы:гречка|food|гречка|гречки|гречки|ж
food:крупы:перловка|food|перловка|перловки|перловки|ж
food:крупы:лапша|food|лапша|лапши|лапши|ж
food:крупы:булгур|food|булгур|булгура|булгур|м
food:крупы:кускус|food|кускус|кускуса|кускус|м
food:крупы:макароны|food|макароны|макарон|макароны|м
# --- белок
food:белок:яйцо|food|яйцо|яйца|яйца|с
food:белок:курица|food|курица|курицы|курицы|ж
food:белок:индейка|food|индейка|индейки|индейки|ж
food:белок:говядина|food|говядина|говядины|говядины|ж
food:белок:свинина|food|свинина|свинины|свинины|ж
food:белок:телятина|food|телятина|телятины|телятины|ж
food:белок:баранина|food|баранина|баранины|баранины|ж
food:белок:оленина|food|оленина|оленины|оленины|ж
food:белок:фарш|food|фарш|фарша|фарш|м
food:белок:котлета|food|котлета|котлеты|котлеты|ж
food:белок:сосиски|food|сосиски|сосисок|сосиски|ж
food:белок:колбаса|food|колбаса|колбасы|колбасы|ж
food:белок:ветчина|food|ветчина|ветчины|ветчины|ж
food:белок:бекон|food|бекон|бекона|беконы|м
food:белок:лосось|food|лосось|лосося|лососи|м
food:белок:тунец|food|тунец|тунца|тунцы|м
food:белок:треска|food|треска|трески|трески|ж
food:белок:судак|food|судак|судака|судаки|м
food:белок:омар|food|омар|омара|омары|м
food:белок:кальмар|food|кальмар|кальмара|кальмары|м
food:белок:креветки|food|креветки|креветок|креветки|ж
food:белок:мидии|food|мидии|мидий|мидии|ж
food:белок:икра|food|икра|икры|икры|ж
# --- молочное и масло
food:молочное:молоко|food|молоко|молока|молоко|с
food:молочное:кефир|food|кефир|кефира|кефиры|м
food:молочное:ряженка|food|ряженка|ряженки|ряженки|ж
food:молочное:простокваша|food|простокваша|простокваши|простокваши|ж
food:молочное:айран|food|айран|айрана|айраны|м
food:молочное:сливки|food|сливки|сливок|сливки|ж
food:молочное:сметана|food|сметана|сметаны|сметаны|ж
food:молочное:йогурт|food|йогурт|йогурта|йогурты|м
food:молочное:творог|food|творог|творога|творог|м
food:молочное:сыр|food|сыр|сыра|сыры|м
food:молочное:моцарелла|food|моцарелла|моцареллы|моцареллы|ж
food:молочное:пармезан|food|пармезан|пармезана|пармезаны|м
food:молочное:фета|food|фета|феты|феты|ж
food:молочное:сливочноемасло|food|сливочное масло|сливочного масла|сливочное масло|с
food:молочное:топленоемасло|food|топлёное масло|топлёного масла|топлёное масло|с
# --- выпечка и сладкое
food:выпечка:хлеб|food|хлеб|хлеба|хлеб|м
food:выпечка:батон|food|батон|батона|батоны|м
food:выпечка:булочка|food|булочка|булочки|булочки|ж
food:выпечка:круассан|food|круассан|круассана|круассаны|м
food:выпечка:багет|food|багет|багета|багеты|м
food:выпечка:лаваш|food|лаваш|лаваша|лаваши|м
food:выпечка:тортилья|food|тортилья|тортильи|тортильи|ж
food:выпечка:печенье|food|печенье|печенья|печенье|с
food:выпечка:вафли|food|вафли|вафель|вафли|ж
food:выпечка:блины|food|блины|блинов|блины|м
food:выпечка:оладьи|food|оладьи|оладий|оладьи|ж
food:выпечка:пончик|food|пончик|пончика|пончики|м
food:выпечка:кекс|food|кекс|кекса|кексы|м
food:выпечка:маффин|food|маффин|маффина|маффины|м
food:выпечка:шарлотка|food|шарлотка|шарлотки|шарлотки|ж
food:выпечка:запеканка|food|запеканка|запеканки|запеканки|ж
food:выпечка:сырники|food|сырники|сырников|сырники|м
food:выпечка:блинчики|food|блинчики|блинчиков|блинчики|м
food:выпечка:пирог|food|пирог|пирога|пироги|м
food:выпечка:торт|food|торт|торта|торты|м
food:выпечка:чизкейк|food|чизкейк|чизкейка|чизкейки|м
food:выпечка:эклер|food|эклер|эклера|эклеры|м
food:выпечка:медовик|food|медовик|медовика|медовики|м
# --- фастфуд и кухни мира
food:фастфуд:пицца|food|пицца|пиццы|пиццы|ж
food:фастфуд:паста|food|паста|пасты|пасты|ж
food:фастфуд:спагетти|food|спагетти|спагетти|спагетти|м
food:фастфуд:лазанья|food|лазанья|лазаньи|лазаньи|ж
food:фастфуд:равиоли|food|равиоли|равиоли|равиоли|м
food:фастфуд:суши|food|суши|суши|суши|м
food:фастфуд:роллы|food|роллы|роллов|роллы|м
food:фастфуд:онигири|food|онигири|онигири|онигири|м
food:фастфуд:бургер|food|бургер|бургера|бургеры|м
food:фастфуд:хотдог|food|хот-дог|хот-дога|хот-доги|м
food:фастфуд:сэндвич|food|сэндвич|сэндвича|сэндвичи|м
food:фастфуд:тако|food|тако|тако|тако|с
food:фастфуд:шаурма|food|шаурма|шаурмы|шаурмы|ж
food:фастфуд:шашлык|food|шашлык|шашлыка|шашлыки|м
food:фастфуд:кебаб|food|кебаб|кебаба|кебабы|м
food:фастфуд:картофельфри|food|картофель фри|картофеля фри|картофель фри|м
food:фастфуд:чипсы|food|чипсы|чипсов|чипсы|ж
food:фастфуд:попкорн|food|попкорн|попкорна|попкорн|м
food:фастфуд:наггетсы|food|наггетсы|наггетсов|наггетсы|м
food:фастфуд:хинкали|food|хинкали|хинкали|хинкали|м
food:фастфуд:цезарь|food|цезарь|цезаря|цезари|м
# --- блюда
food:блюдо:борщ|food|борщ|борща|борщи|м
food:блюдо:щи|food|щи|щей|щи|ж
food:блюдо:окрошка|food|окрошка|окрошки|окрошки|ж
food:блюдо:холодник|food|холодник|холодника|холодники|м
food:блюдо:солянка|food|солянка|солянки|солянки|ж
food:блюдо:рассольник|food|рассольник|рассольника|рассольники|м
food:блюдо:уха|food|уха|ухи|ухи|ж
food:блюдо:харчо|food|харчо|харчо|харчо|ж
food:блюдо:плов|food|плов|плова|пловы|м
food:блюдо:бефстроганов|food|бефстроганов|бефстроганова|бефстрогановы|м
food:блюдо:голубцы|food|голубцы|голубцов|голубцы|м
food:блюдо:холодец|food|холодец|холодца|холодцы|м
food:блюдо:заливное|food|заливное|заливного|заливные|с
food:блюдо:оливье|food|салат оливье|салата оливье|салаты оливье|м
food:блюдо:греческий|food|греческий салат|греческого салата|греческие салаты|м
# --- напитки, сладости, приправы
food:напиток:чай|food|чай|чая|чаи|м
food:напиток:кофе|food|кофе|кофе|кофе|м
food:напиток:какао|food|какао|какао|какао|с
food:напиток:сок|food|сок|сока|соки|м
food:напиток:компот|food|компот|компота|компоты|м
food:напиток:морс|food|морс|морса|морсы|м
food:напиток:квас|food|квас|кваса|квасы|м
food:напиток:лимонад|food|лимонад|лимонада|лимонады|м
food:напиток:сироп|food|сироп|сиропа|сиропы|м
food:напиток:мёд|food|мёд|мёда|мёды|м
food:напиток:варенье|food|варенье|варенья|варенье|с
food:напиток:джем|food|джем|джема|джемы|м
food:напиток:повидло|food|повидло|повидла|повидло|с
food:приправа:сахар|food|сахар|сахара|сахар|м
food:приправа:соль|food|соль|соли|соль|ж
food:приправа:соевыйсоус|food|соевый соус|соевого соуса|соевые соусы|м
food:приправа:кетчуп|food|кетчуп|кетчупа|кетчупы|м
food:приправа:майонез|food|майонез|майонеза|майонезы|м
food:приправа:горчица|food|горчица|горчицы|горчицы|ж
food:приправа:аджика|food|аджика|аджики|аджики|ж
food:приправа:корица|food|корица|корицы|корицы|ж
food:приправа:гвоздика|food|гвоздика|гвоздики|гвоздики|ж
food:приправа:ваниль|food|ваниль|ванили|ванили|ж
food:приправа:кориандр|food|кориандр|кориандра|кориандр|м
food:приправа:мята|food|мята|мяты|мяты|ж
food:приправа:кардамон|food|кардамон|кардамона|кардамоны|м
"""

In [ ]:
NOT_FOOD_LEX = r"""
# --- мебель
nf:мебель:стул|not_food|стул|стула|стулья|м
nf:мебель:табурет|not_food|табурет|табурета|табуреты|м
nf:мебель:стол|not_food|стол|стола|столы|м
nf:мебель:диван|not_food|диван|дивана|диваны|м
nf:мебель:кресло|not_food|кресло|кресла|кресла|с
nf:мебель:шкаф|not_food|шкаф|шкафа|шкафы|м
nf:мебель:комод|not_food|комод|комода|комоды|м
nf:мебель:тумба|not_food|тумба|тумбы|тумбы|ж
nf:мебель:полка|not_food|полка|полки|полки|ж
nf:мебель:кровать|not_food|кровать|кровати|кровати|ж
nf:мебель:матрас|not_food|матрас|матраса|матрасы|м
nf:мебель:зеркало|not_food|зеркало|зеркала|зеркала|с
nf:мебель:картина|not_food|картина|картины|картины|ж
nf:мебель:ковёр|not_food|ковёр|ковра|ковры|м
nf:мебель:стеллаж|not_food|стеллаж|стеллажа|стеллажи|м
nf:мебель:настенныечасы|not_food|настенные часы|настенных часов|настенные часы|м
nf:свет:люстра|not_food|люстра|люстры|люстры|ж
nf:свет:торшер|not_food|торшер|торшера|торшеры|м
nf:свет:настольнаялампа|not_food|настольная лампа|настольной лампы|настольные лампы|ж
nf:свет:подсветка|not_food|подсветка|подсветки|подсветки|ж
# --- техника
nf:техника:ноутбук|not_food|ноутбук|ноутбука|ноутбуки|м
nf:техника:смартфон|not_food|смартфон|смартфона|смартфоны|м
nf:техника:планшет|not_food|планшет|планшета|планшеты|м
nf:техника:телевизор|not_food|телевизор|телевизора|телевизоры|м
nf:техника:принтер|not_food|принтер|принтера|принтеры|м
nf:техника:наушники|not_food|наушники|наушников|наушники|м
nf:техника:колонка|not_food|колонка|колонки|колонки|ж
nf:техника:камера|not_food|камера|камеры|камеры|ж
nf:техника:микрофон|not_food|микрофон|микрофона|микрофоны|м
nf:техника:компьютернаямышь|not_food|компьютерная мышь|компьютерной мыши|компьютерные мыши|ж
nf:техника:клавиатура|not_food|клавиатура|клавиатуры|клавиатуры|ж
nf:техника:роутер|not_food|роутер|роутера|роутеры|м
nf:техника:проектор|not_food|проектор|проектора|проекторы|м
nf:техника:пылесос|not_food|пылесос|пылесоса|пылесосы|м
nf:техника:стиральнаямашина|not_food|стиральная машина|стиральной машины|стиральные машины|ж
nf:техника:холодильник|not_food|холодильник|холодильника|холодильники|м
nf:техника:микроволновка|not_food|микроволновая печь|микроволновой печи|микроволновые печи|ж
nf:техника:электрическийчайник|not_food|электрический чайник|электрического чайника|электрические чайники|м
nf:техника:кофемашина|not_food|кофемашина|кофемашины|кофемашины|ж
nf:техника:тостер|not_food|тостер|тостера|тостеры|м
nf:техника:блендер|not_food|блендер|блендера|блендеры|м
nf:техника:мультиварка|not_food|мультиварка|мультиварки|мультиварки|ж
nf:техника:аэрогриль|not_food|аэрогриль|аэрогриля|аэрогрили|м
nf:техника:посудомоечнаямашина|not_food|посудомоечная машина|посудомоечной машины|посудомоечные машины|ж
nf:техника:кондиционер|not_food|кондиционер|кондиционера|кондиционеры|м
nf:техника:обогреватель|not_food|обогреватель|обогревателя|обогреватели|м
nf:техника:видеокарта|not_food|видеокарта|видеокарты|видеокарты|ж
nf:техника:сервер|not_food|сервер|сервера|серверы|м
# --- кухонный инвентарь (не еда!)
nf:инвентарь:кастрюля|not_food|кастрюля|кастрюли|кастрюли|ж
nf:инвентарь:сковорода|not_food|сковорода|сковороды|сковороды|ж
nf:инвентарь:кухонныйнож|not_food|кухонный нож|кухонного ножа|кухонные ножи|м
nf:инвентарь:разделочнаядоска|not_food|разделочная доска|разделочной доски|разделочные доски|ж
nf:инвентарь:вилка|not_food|вилка|вилки|вилки|ж
nf:инвентарь:ложка|not_food|ложка|ложки|ложки|ж
nf:инвентарь:тарелка|not_food|тарелка|тарелки|тарелки|ж
nf:инвентарь:чашка|not_food|чашка|чашки|чашки|ж
nf:инвентарь:блюдо|not_food|праздничное блюдо|праздничного блюда|праздничные блюда|с
nf:инвентарь:стакан|not_food|стакан|стакана|стаканы|м
nf:инвентарь:кружка|not_food|кружка|кружки|кружки|ж
nf:инвентарь:банка|not_food|банка|банки|банки|ж
nf:инвентарь:бутылка|not_food|бутылка|бутылки|бутылки|ж
nf:инвентарь:термос|not_food|термос|термоса|термосы|м
nf:инвентарь:поднос|not_food|поднос|подноса|подносы|м
nf:инвентарь:дуршлаг|not_food|дуршлаг|дуршлага|дуршлаги|м
nf:инвентарь:кухонныйвенчик|not_food|кухонный венчик|кухонного венчика|кухонные венчики|м
nf:инвентарь:сито|not_food|сито|сита|сита|с
nf:инвентарь:салфетка|not_food|салфетка|салфетки|салфетки|ж
nf:инвентарь:противень|not_food|противень|противня|противни|м
nf:инвентарь:сотейник|not_food|сотейник|сотейника|сотейники|м
# --- одежда
nf:одежда:куртка|not_food|куртка|куртки|куртки|ж
nf:одежда:джинсы|not_food|джинсы|джинсов|джинсы|м
nf:одежда:свитер|not_food|свитер|свитера|свитера|м
nf:одежда:шапка|not_food|шапка|шапки|шапки|ж
nf:одежда:шарф|not_food|шарф|шарфа|шарфы|м
nf:одежда:перчатки|not_food|перчатки|перчаток|перчатки|ж
nf:одежда:ботинки|not_food|ботинки|ботинок|ботинки|м
nf:одежда:кроссовки|not_food|кроссовки|кроссовок|кроссовки|м
nf:одежда:туфли|not_food|туфли|туфель|туфли|ж
nf:одежда:сандалии|not_food|сандалии|сандалий|сандалии|ж
nf:одежда:носки|not_food|носки|носков|носки|м
nf:одежда:ремень|not_food|ремень|ремня|ремни|м
nf:одежда:платье|not_food|платье|платья|платья|с
nf:одежда:халат|not_food|халат|халата|халаты|м
nf:одежда:тапки|not_food|тапки|тапок|тапки|ж
nf:одежда:зонт|not_food|зонт|зонта|зонты|м
# --- аксессуары и уход
nf:аксессуары:сумка|not_food|сумка|сумки|сумки|ж
nf:аксессуары:рюкзак|not_food|рюкзак|рюкзака|рюкзаки|м
nf:аксессуары:чемодан|not_food|чемодан|чемодана|чемоданы|м
nf:аксессуары:кошелёк|not_food|кошелёк|кошелька|кошельки|м
nf:аксессуары:ключиотмашины|not_food|ключи от машины|ключей от машины|ключи от машины|м
nf:аксессуары:солнцезащитныеочки|not_food|солнцезащитные очки|солнцезащитных очков|солнцезащитные очки|м
nf:аксессуары:наручныечасы|not_food|наручные часы|наручных часов|наручные часы|м
nf:аксессуары:браслет|not_food|браслет|браслета|браслеты|м
nf:аксессуары:кольцо|not_food|кольцо|кольца|кольца|с
nf:аксессуары:серьги|not_food|серьги|серёг|серьги|ж
nf:аксессуары:ожерелье|not_food|ожерелье|ожерелья|ожерелья|с
nf:аксессуары:расчёска|not_food|расчёска|расчёски|расчёски|ж
nf:аксессуары:зубнаящётка|not_food|зубная щётка|зубной щётки|зубные щётки|ж
nf:аксессуары:футляр|not_food|футляр|футляра|футляры|м
# --- бытовая химия
nf:химия:мыло|not_food|мыло|мыла|мыла|с
nf:химия:шампунь|not_food|шампунь|шампуня|шампуни|м
nf:химия:кремдлярук|not_food|крем для рук|крема для рук|кремы для рук|м
nf:химия:стиральныйпорошок|not_food|стиральный порошок|стирального порошка|стиральные порошки|м
nf:химия:чистящеесредство|not_food|чистящее средство|чистящего средства|чистящие средства|с
nf:химия:лакданогтей|not_food|лак для ногтей|лака для ногтей|лаки для ногтей|м
nf:химия:краска|not_food|краска|краски|краски|ж
nf:химия:изолента|not_food|изолента|изоленты|изоленты|ж
nf:химия:скотч|not_food|скотч|скотча|скотчи|м
nf:химия:клей|not_food|клей|клея|клеи|м
nf:химия:средствоотнасекомых|not_food|средство от насекомых|средства от насекомых|средства от насекомых|с
# --- инструменты
nf:инструменты:молоток|not_food|молоток|молотка|молотки|м
nf:инструменты:отвёртка|not_food|отвёртка|отвёртки|отвёртки|ж
nf:инструменты:гаечныйключ|not_food|гаечный ключ|гаечного ключа|гаечные ключи|м
nf:инструменты:пила|not_food|пила|пилы|пилы|ж
nf:инструменты:дрель|not_food|дрель|дрели|дрели|ж
nf:инструменты:шуруповёрт|not_food|шуруповёрт|шуруповёрта|шуруповёрты|м
nf:инструменты:наборсвёрл|not_food|набор свёрл|набора свёрл|наборы свёрл|м
nf:инструменты:гвозди|not_food|гвозди|гвоздей|гвозди|м
nf:инструменты:шурупы|not_food|шурупы|шурупов|шурупы|м
nf:инструменты:рулетка|not_food|рулетка|рулетки|рулетки|ж
nf:инструменты:уровень|not_food|уровень|уровня|уровни|м
nf:инструменты:лестница|not_food|лестница|лестницы|лестницы|ж
nf:инструменты:тачка|not_food|тачка|тачки|тачки|ж
nf:инструменты:лопата|not_food|лопата|лопаты|лопаты|ж
nf:инструменты:грабли|not_food|грабли|граблей|грабли|ж
nf:инструменты:лейка|not_food|лейка|лейки|лейки|ж
nf:инструменты:секатор|not_food|секатор|секатора|секаторы|м
nf:инструменты:бензопила|not_food|бензопила|бензопилы|бензопилы|ж
nf:инструменты:газонокосилка|not_food|газонокосилка|газонокосилки|газонокосилки|ж
nf:инструменты:шланг|not_food|шланг|шланга|шланги|м
nf:инструменты:тиски|not_food|тиски|тисков|тиски|м
nf:инструменты:наждачка|not_food|наждачная бумага|наждачной бумаги|наждачная бумага|ж
# --- дача и улица
nf:дача:горшок|not_food|горшок|горшка|горшки|м
nf:дача:ваза|not_food|ваза|вазы|вазы|ж
nf:дача:кашпо|not_food|кашпо|кашпо|кашпо|с
nf:дача:уличныйфонарь|not_food|уличный фонарь|уличного фонаря|уличные фонари|м
nf:дача:скамейка|not_food|скамейка|скамейки|скамейки|ж
nf:дача:качели|not_food|качели|качелей|качели|ж
nf:дача:детскаягорка|not_food|детская горка|детской горки|детские горки|ж
nf:дача:песочница|not_food|песочница|песочницы|песочницы|ж
nf:дача:мяч|not_food|мяч|мяча|мячи|м
nf:дача:велосипед|not_food|велосипед|велосипеда|велосипеды|м
nf:дача:самокат|not_food|самокат|самоката|самокаты|м
nf:дача:скейтборд|not_food|скейтборд|скейтборда|скейтборды|м
nf:дача:роликовыеконьки|not_food|роликовые коньки|роликовых коньков|роликовые коньки|м
nf:дача:мангал|not_food|мангал|мангала|мангалы|м
nf:дача:теплица|not_food|теплица|теплицы|теплицы|ж
nf:дача:пруд|not_food|пруд|пруда|пруды|м
nf:дача:садовыйбордюр|not_food|садовый бордюр|садового бордюра|садовые бордюры|м
# --- транспорт
nf:транспорт:легковаямашина|not_food|легковая машина|легковой машины|легковые машины|ж
nf:транспорт:автобус|not_food|автобус|автобуса|автобусы|м
nf:транспорт:трамвай|not_food|трамвай|трамвая|трамваи|м
nf:транспорт:метро|not_food|метро|метро|метро|с
nf:транспорт:поезд|not_food|поезд|поезда|поезда|м
nf:транспорт:самолёт|not_food|самолёт|самолёта|самолёты|м
nf:транспорт:корабль|not_food|корабль|корабля|корабли|м
nf:транспорт:яхта|not_food|яхта|яхты|яхты|ж
nf:транспорт:вертолёт|not_food|вертолёт|вертолёта|вертолёты|м
nf:транспорт:ракета|not_food|ракета|ракеты|ракеты|ж
nf:транспорт:воздушныйшар|not_food|воздушный шар|воздушного шара|воздушные шары|м
nf:транспорт:прицеп|not_food|прицеп|прицепа|прицепы|м
nf:транспорт:скутер|not_food|скутер|скутера|скутеры|м
nf:транспорт:лодка|not_food|лодка|лодки|лодки|ж
nf:транспорт:катер|not_food|катер|катера|катера|м
nf:транспорт:трактор|not_food|трактор|трактора|тракторы|м
# --- офис и канцелярия
nf:офис:книга|not_food|книга|книги|книги|ж
nf:офис:тетрадь|not_food|тетрадь|тетради|тетради|ж
nf:офис:ручка|not_food|ручка|ручки|ручки|ж
nf:офис:карандаш|not_food|карандаш|карандаша|карандаши|м
nf:офис:папка|not_food|папка|папки|папки|ж
nf:офис:блокнот|not_food|блокнот|блокнота|блокноты|м
nf:офис:бумага|not_food|бумага|бумаги|бумаги|ж
nf:офис:ластик|not_food|ластик|ластика|ластики|м
nf:офис:дневник|not_food|дневник|дневника|дневники|м
nf:офис:флешка|not_food|флешка|флешки|флешки|ж
nf:офис:кабель|not_food|кабель|кабеля|кабели|м
nf:офис:картридж|not_food|картридж|картриджа|картриджи|м
# --- стройка и сантехника
nf:стройка:кирпич|not_food|кирпич|кирпича|кирпичи|м
nf:стройка:цемент|not_food|цемент|цемента|цемента|м
nf:стройка:арматура|not_food|арматура|арматуры|арматуры|ж
nf:стройка:профнастил|not_food|профнастил|профнастила|профнастилы|м
nf:стройка:брус|not_food|брус|бруса|брусья|м
nf:стройка:фанера|not_food|фанера|фанеры|фанеры|ж
nf:стройка:гипсокартон|not_food|гипсокартон|гипсокартона|гипсокартоны|м
nf:стройка:плитка|not_food|плитка|плитки|плитки|ж
nf:стройка:линолеум|not_food|линолеум|линолеума|линолеумы|м
nf:сантехника:кран|not_food|кран|крана|краны|м
nf:сантехника:смеситель|not_food|смеситель|смесителя|смесители|м
nf:сантехника:труба|not_food|труба|трубы|трубы|ж
nf:сантехника:раковина|not_food|раковина|раковины|раковины|ж
nf:сантехника:ванна|not_food|ванна|ванны|ванны|ж
nf:сантехника:унитаз|not_food|унитаз|унитаза|унитазы|м
nf:сантехника:счётчикводы|not_food|счётчик воды|счётчика воды|счётчики воды|м
# --- текстиль, музыка, спорт, животные, медицина
nf:текстиль:штора|not_food|штора|шторы|шторы|ж
nf:текстиль:покрывало|not_food|покрывало|покрывала|покрывала|с
nf:текстиль:полотенце|not_food|полотенце|полотенца|полотенца|с
nf:музыка:гитара|not_food|гитара|гитары|гитары|ж
nf:музыка:скрипка|not_food|скрипка|скрипки|скрипки|ж
nf:музыка:пианино|not_food|пианино|пианино|пианино|с
nf:музыка:барабан|not_food|барабан|барабана|барабаны|м
nf:музыка:флейта|not_food|флейта|флейты|флейты|ж
nf:спорт:гантель|not_food|гантель|гантели|гантели|ж
nf:спорт:велотренажёр|not_food|велотренажёр|велотренажёра|велотренажёры|м
nf:спорт:лыжи|not_food|лыжи|лыж|лыжи|ж
nf:спорт:санки|not_food|санки|санок|санки|ж
nf:спорт:турник|not_food|турник|турника|турники|м
nf:животные:кошка|not_food|кошка|кошки|кошки|ж
nf:животные:собака|not_food|собака|собаки|собаки|ж
nf:животные:аквариум|not_food|аквариум|аквариума|аквариумы|м
nf:животные:клетка|not_food|клетка|клетки|клетки|ж
nf:животные:поводок|not_food|поводок|поводка|поводки|м
nf:животные:кормушка|not_food|кормушка|кормушки|кормушки|ж
nf:медицина:шприц|not_food|шприц|шприца|шприцы|м
nf:медицина:тонометр|not_food|тонометр|тонометра|тонометры|м
"""

In [ ]:
VALID_LABELS = set(ID2LABEL.values())
VALID_GENDERS = {"м", "ж", "с"}


def parse_lex(raw: str) -> list[dict]:
    """'группа|метка|nom|gen|pl|род' -> список словарей. Любая ошибка падает сразу."""
    out = []
    for line_no, line in enumerate(raw.strip().splitlines(), start=1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) != 6:
            raise ValueError(
                f"строка {line_no}: ожидалось 6 полей 'группа|метка|nom|gen|pl|род', "
                f"получено {len(parts)}:\n  {line}"
            )
        group, label, nom, gen, pl, gender = parts
        if label not in VALID_LABELS:
            raise ValueError(f"строка {line_no}: метка {label!r} не из {VALID_LABELS}")
        if gender not in VALID_GENDERS:
            raise ValueError(f"строка {line_no}: род {gender!r} не из {VALID_GENDERS}")
        for field, value in (("nom", nom), ("gen", gen), ("pl", pl)):
            if not value or value == "-":
                raise ValueError(f"строка {line_no}: пустое поле {field}")
        if not group:
            raise ValueError(f"строка {line_no}: пустая группа")
        out.append({"group": group, "label": label, "nom": nom, "gen": gen, "pl": pl,
                    "gender": gender, "first_word": nom.split()[0].lower()})
    return out


FOOD_ENTITIES = parse_lex(FOOD_LEX)
NOT_FOOD_ENTITIES = parse_lex(NOT_FOOD_LEX)
ENTITIES = FOOD_ENTITIES + NOT_FOOD_ENTITIES

# проверки таблицы: дубли групп, одна метка на группу, одна и та же лексема в обоих классах
_group_labels = defaultdict(set)
for _e in ENTITIES:
    _group_labels[_e["group"]].add(_e["label"])
_dup_groups = {g: sorted(v) for g, v in _group_labels.items() if len(v) > 1}
assert not _dup_groups, f"у группы несколько меток: {_dup_groups}"

_counts = Counter(e["group"] for e in ENTITIES)
_dups = [g for g, c in _counts.items() if c > 1]
assert not _dups, f"группа встречается дважды: {_dups}"

_noms = defaultdict(set)
for _e in ENTITIES:
    _noms[_e["nom"].lower()].add(_e["label"])
_cross = {n: sorted(v) for n, v in _noms.items() if len(v) > 1}
assert not _cross, (
    f"одна и та же лексема в обоих классах — это неоднозначность, её надо решать "
    f"в hard-пакете, а не в лексике: {_cross}"
)

print(f"food-сущностей:     {len(FOOD_ENTITIES)}")
print(f"not_food-сущностей: {len(NOT_FOOD_ENTITIES)}")
print(f"групп:              {len(_counts)}")

## 3. Фреймы и слоты

Ключевая идея: **часть фреймов общая для обоих классов**. Если еда и не-еда собираются одним и тем же
синтаксическим каркасом, метку уже нельзя вывести из формы предложения — приходится смотреть
на смысл. Общий и пул `surf` («на столе» встречается в обоих классах), и набор прилагательных
(«мягкий» есть у обоих).

Прилагательные заданы **по роду** (`ADJ[label][gender]`), поэтому согласование не может
сломаться: код не склеивает окончания, он берёт готовую форму.

In [ ]:
SURF = [
    "столе", "тумбе", "полке", "подоконнике", "кухне", "мастерской", "сетке",
    "коробке", "сумке", "прилавке", "складе", "руках",
]

# --- прилагательные по роду: согласование не собирается из окончаний, а берётся готовым ---
ADJ = {
    "food": {
        "м": ["мягкий", "зелёный", "спелый", "варёный", "сладкий", "свежий"],
        "ж": ["мягкая", "зелёная", "спелая", "варёная", "сладкая", "свежая"],
        "с": ["мягкое", "зелёное", "спелое", "варёное", "сладкое", "свежее"],
    },
    "not_food": {
        "м": ["мягкий", "новый", "старый", "деревянный", "тяжёлый", "лёгкий", "тёплый", "прочный"],
        "ж": ["мягкая", "новая", "старая", "деревянная", "тяжёлая", "лёгкая", "тёплая", "прочная"],
        "с": ["мягкое", "новое", "старое", "деревянное", "тяжёлое", "лёгкое", "тёплое", "прочное"],
    },
}

# --- предикативные качества по роду (идут после запятой, как «Яблоко, свежее, на столе.») ---
QUAL = {
    "food": {
        "м": ["свежий", "спелый", "крупный", "вкусный", "мороженый", "домашний", "варёный",
              "жареный", "сладкий", "мягкий"],
        "ж": ["свежая", "спелая", "крупная", "вкусная", "мороженая", "домашняя", "варёная",
              "жареная", "сладкая", "мягкая"],
        "с": ["свежее", "спелое", "крупное", "вкусное", "мороженое", "домашнее", "варёное",
              "жареное", "сладкое", "мягкое"],
    },
    "not_food": {
        "м": ["новый", "старый", "деревянный", "металлический", "тяжёлый", "лёгкий", "тёплый",
              "холодный", "прочный", "мягкий", "сладкий"],
        "ж": ["новая", "старая", "деревянная", "металлическая", "тяжёлая", "лёгкая", "тёплая",
              "холодная", "прочная", "мягкая", "сладкая"],
        "с": ["новое", "старое", "деревянное", "металлическое", "тяжёлое", "лёгкое", "тёплое",
              "холодное", "прочное", "мягкое", "сладкое"],
    },
}

MEALS = ["завтрак", "обед", "ужин", "полдник", "перекус", "праздник", "поездка", "дача", "офис"]
FOOD_ACTS = ["жарить", "варить", "запекать", "тушить", "мариновать", "нарезать", "очистить", "взбить"]
ROOMS = ["гостиной", "кухне", "спальне", "кабинете", "мастерской", "даче", "офисе", "гараже"]
NF_ACTS = ["починить", "собрать", "отремонтировать", "заточить", "подключить", "покрасить", "заменить"]

# --- ОБЩИЕ фреймы: ровно один и тот же каркас для обоих классов ---
SHARED_FRAMES = [
    "{Nom}, {qual}, на {surf}.",
    "На {surf} лежат {pl}.",
    "на {surf} — {nom} и ещё несколько {pl}.",
    "{adj} {nom} на {surf}.",
    "Много {pl} на {surf}.",
    "Фото: {nom}.",
    "Описание: {nom}, {qual}.",
    "Сколько стоит {nom}?",
    "Где купить {nom}?",
    "Купить {nom} к празднику",
    "Список покупок: {nom}, {nom2}, {nom3}.",
    "Не знаю, что выбрать: {nom} или {nom2}.",
    "Хочу {nom}.",
    "Скидка на {nom} до конца недели.",
    "Что это такое? {Nom}",
    "Расскажите про {nom}.",
    "В упаковке: {nom} и {nom2}.",
    "Обсуждаем {nom} в чате.",
    "Показать {nom} другу.",
    "Найти в интернете: {nom}",
]

# --- фреймы, специфичные для класса ---
FOOD_FRAMES = [
    "Рецепт {gen}: {act} {nom}, добавить {nom2}.",
    "Для {meal} приготовить {nom}.",
    "Ингредиенты: {nom}, {nom2}, {nom3}.",
    "Меню: {nom}, {nom2} и {nom3}.",
    "Сколько калорий в {nom}?",
    "Как правильно приготовить {nom}?",
    "Диета: нельзя {nom}, можно {nom2}.",
    "Подают {nom} с {nom2}.",
    "Холодильник: {nom}, {nom2}, {nom3}.",
    "Еда в дорогу: {nom} и {nom2}.",
    "В магазине закончились {pl}.",
    "Надо докупить {nom}.",
    "Что приготовить из {gen}?",
    "Гарнир — {nom}.",
    "Заказали {nom} и {nom2} через приложение.",
    "Отзыв: {nom} — вкусно.",
    "{meal}: {nom} и {nom2}.",
]

NOT_FOOD_FRAMES = [
    "Сломался {nom}, надо чинить.",
    "Инструмент в {room}: {nom}.",
    "Инструкция: как собрать {nom} из {nom2}.",
    "Стоит на {surf} в {room}.",
    "Купил {nom} для {room}.",
    "Как ухаживать за {gen}?",
    "Список для мастерской: {nom}, {nom2}, {nom3}.",
    "Заказал {nom} в интернет-магазине.",
    "Нужно {act} {nom} перед ремонтом.",
    "Занимает всю полку в {room}.",
    "Стоимость {gen} выросла.",
    "Фото: {adj} {nom} в {room}.",
    "Что делать, если сломался {nom}?",
    "В коробке лежат {pl}.",
    "Выбираю между {nom} и {nom2}.",
    "Собрал {nom} за вечер.",
    "Отзыв: {nom} — на пять звёзд.",
]

FRAME_SETS = {"food": SHARED_FRAMES + FOOD_FRAMES, "not_food": SHARED_FRAMES + NOT_FOOD_FRAMES}

# все слоты, которые вообще встречаются во фреймах, — контекст рендера заполняется целиком,
# поэтому KeyError невозможен даже если фрейм пополнится новым слотом
CONTEXT_KEYS = ["nom", "Nom", "gen", "pl", "adj", "qual", "surf", "room", "meal", "act",
                "nom2", "nom3"]

_shared_slots = {m.group(1) for f in SHARED_FRAMES for m in re.finditer(r"\{(\w+)\}", f)}
print(f"общих фреймов: {len(SHARED_FRAMES)}, фреймов food: {len(FOOD_FRAMES)}, "
      f"фреймов not_food: {len(NOT_FOOD_FRAMES)}")
print(f"слоты в общих фреймах: {sorted(_shared_slots)}")
assert _shared_slots <= set(CONTEXT_KEYS), "в общих фреймах есть слот вне CONTEXT_KEYS"

## 4. Хард-пакет

Ручной набор, который закрывает то, чего в лексике быть не может: перенос значения, отрицание,
метафору, жанровые сдвиги и «съедобно-звучное несъедобное».

Группа = `слово + значение` (`hard:ff:паста:food` и `hard:ff:паста:not_food` — разные группы).
Это сделано намеренно: одно и то же слово с противоположными метками в разных сплитах — это и есть
проверка на различение значений, а не утечка. Если смешать их в одной группе, задача на тесте
схлопнется обратно в словарь.

Каждую строку можно править руками: это обычный Python-список, а не замороженный артефакт.

In [ ]:
# (текст, метка, группа)
HARD_PACK = [
    # ---------- ложные друзья: слово есть, значения разные ----------
    ("Паста с морепродуктами — классика.", "food", "hard:ff:паста:food"),
    ("Сварил пасту с томатным соусом.", "food", "hard:ff:паста:food"),
    ("Рецепт пасты карбонара.", "food", "hard:ff:паста:food"),
    ("Добавь пасту в кипящую воду за девять минут.", "food", "hard:ff:паста:food"),
    ("Купил зубную пасту, щётка мягкая.", "not_food", "hard:ff:паста:not_food"),
    ("Паста для стен в тюбике.", "not_food", "hard:ff:паста:not_food"),
    ("Малярная паста трещит по штукатурке.", "not_food", "hard:ff:паста:not_food"),
    ("Перец чили, режу аккуратно.", "food", "hard:ff:перец:food"),
    ("Болгарский перец в салат.", "food", "hard:ff:перец:food"),
    ("Молотый перец в суп.", "food", "hard:ff:перец:food"),
    ("Перец пришёл в кабинет новым сотрудником.", "not_food", "hard:ff:перец:not_food"),
    ("Перец опоздал на планёрку.", "not_food", "hard:ff:перец:not_food"),
    ("Наш перец — лучший в отделе.", "not_food", "hard:ff:перец:not_food"),
    ("Горчица острая, в стеклянной банке.", "food", "hard:ff:горчица:food"),
    ("Горчичный соус к мясу.", "food", "hard:ff:горчица:food"),
    ("Горчица по поводу отпуска: едем в августе.", "not_food", "hard:ff:горчица:not_food"),
    ("Хрен ему горчица, что ли.", "not_food", "hard:ff:горчица:not_food"),
    ("Хрен столовый, остро.", "food", "hard:ff:хрен:food"),
    ("Натрите хрен на тёрке.", "food", "hard:ff:хрен:food"),
    ("Хрен с ним, что тут обсуждать.", "not_food", "hard:ff:хрен:not_food"),
    ("Соль с чесноком и перцем.", "food", "hard:ff:соль:food"),
    ("Щепотка соли в суп.", "food", "hard:ff:соль:food"),
    ("Соль оказалась в рецепте дважды.", "food", "hard:ff:соль:food"),
    ("Соль в оркестре — дирижёрский ход.", "not_food", "hard:ff:соль:not_food"),
    ("Соль на скипке, а не на паузе.", "not_food", "hard:ff:соль:not_food"),
    ("Взять соль в пункте.", "not_food", "hard:ff:соль:not_food"),
    ("Сливочное масло в холодильнике.", "food", "hard:ff:масло:food"),
    ("Оливковое масло для заправки.", "food", "hard:ff:масло:food"),
    ("Использовать масло для жарки.", "food", "hard:ff:масло:food"),
    ("Масляная картина в позолоченной раме.", "not_food", "hard:ff:масло:not_food"),
    ("Набор масляных красок.", "not_food", "hard:ff:масло:not_food"),
    ("Масло для пропитки тканей.", "not_food", "hard:ff:масло:not_food"),
    ("Сливочный крем на бисквит.", "food", "hard:ff:крем:food"),
    ("Жидкий крем в банке.", "food", "hard:ff:крем:food"),
    ("Приготовить крем для торта.", "food", "hard:ff:крем:food"),
    ("Крем для обуви чёрный.", "not_food", "hard:ff:крем:not_food"),
    ("Крем под глаза от морщин.", "not_food", "hard:ff:крем:not_food"),
    ("Молоко в бутылке, 3,2%.", "food", "hard:ff:молоко:food"),
    ("Молоко ушло за день.", "food", "hard:ff:молоко:food"),
    ("Молоко-трава расцвела на подоконнике.", "not_food", "hard:ff:молоко:not_food"),
    ("Герань молоко в кашпо.", "not_food", "hard:ff:молоко:not_food"),
    ("Свежие яблоки на тарелке.", "food", "hard:ff:яблоко:food"),
    ("Яблоки в шарлотке.", "food", "hard:ff:яблоко:food"),
    ("Яблочный сок на завтрак.", "food", "hard:ff:яблоко:food"),
    ("Apple представила новый процессор.", "not_food", "hard:ff:яблоко:not_food"),
    ("Яблоко от яблони недалеко падает.", "not_food", "hard:ff:яблоко:not_food"),
    ("Проткнули как яблоко.", "not_food", "hard:ff:яблоко:not_food"),
    ("Капуста романская на столе.", "food", "hard:ff:капуста:food"),
    ("Ватрушка с капустой.", "food", "hard:ff:капуста:food"),
    ("Опер пела Капусту в «Евгении Онегине».", "not_food", "hard:ff:капуста:not_food"),
    ("Сорок лет в одной капусте.", "not_food", "hard:ff:капуста:not_food"),
    ("Лимон, зелёный, нарезанный дольками.", "food", "hard:ff:лимон:food"),
    ("Лимонный сок в коктейле.", "food", "hard:ff:лимон:food"),
    ("Классический лимон, чёрный, с кожаным салоном.", "not_food", "hard:ff:лимон:not_food"),
    ("Груша, спелая, с соком.", "food", "hard:ff:груша:food"),
    ("Грушовая начинка для пирога.", "food", "hard:ff:груша:food"),
    ("Галстук-бабочка «груша» в чёрном галстуке.", "not_food", "hard:ff:груша:not_food"),
    ("Мятный чай после ужина.", "food", "hard:ff:мята:food"),
    ("Листья мяты в коктейле.", "food", "hard:ff:мята:food"),
    ("Четыре мяты в кошельке.", "not_food", "hard:ff:мята:not_food"),
    ("Отдал две мяты за долг.", "not_food", "hard:ff:мята:not_food"),
    ("Лук зелёный, нарезанный кольцами.", "food", "hard:ff:лук:food"),
    ("Суп с луком.", "food", "hard:ff:лук:food"),
    ("Лук без тетивы висит на стене.", "not_food", "hard:ff:лук:not_food"),
    ("Отдал лук на сохранение другу.", "not_food", "hard:ff:лук:not_food"),
    ("Гвоздика пряная, молотый.", "food", "hard:ff:гвоздика:food"),
    ("В блюде гвоздика и корица.", "food", "hard:ff:гвоздика:food"),
    ("Гвоздика альбери в горшке.", "not_food", "hard:ff:гвоздика:not_food"),
    ("Мороженое в вафельном рожке.", "food", "hard:ff:мороженое:food"),
    ("Мороженое тает на солнце.", "food", "hard:ff:мороженое:food"),
    ("Не будь ребёнком, ты не мороженое.", "not_food", "hard:ff:мороженое:not_food"),
    ("Грецкий орех в салат.", "food", "hard:ff:орех:food"),
    ("Ореховый соус к мясу.", "food", "hard:ff:орех:food"),
    ("Крепкий орех из отряда.", "not_food", "hard:ff:орех:not_food"),
    ("Ещё один орех в новом релизе.", "not_food", "hard:ff:орех:not_food"),
    ("Бананы в кассе.", "food", "hard:ff:банан:food"),
    ("Банановый кетчуп к наггетсам.", "food", "hard:ff:банан:food"),
    ("Сдувшийся банан, ноль энергии.", "not_food", "hard:ff:банан:not_food"),
    ("Петрушка мелко нарезана.", "food", "hard:ff:петрушка:food"),
    ("Петрушка в горшке на балконе.", "food", "hard:ff:петрушка:food"),
    ("Балерина Петрушка в белой пачке.", "not_food", "hard:ff:петрушка:not_food"),
    # ---------- метафорические значения ----------
    ("Рецепт успеха никто не выдаёт.", "not_food", "hard:metaphor:рецепт"),
    ("Кулинарный шедевр отдела продаж.", "not_food", "hard:metaphor:кулинарный"),
    ("Проект кипит, нужен дедлайн.", "not_food", "hard:metaphor:кипит"),
    ("Подогреть отношения с клиентом.", "not_food", "hard:metaphor:подогреть"),
    ("Собрать с нуля за неделю.", "not_food", "hard:metaphor:собрать"),
    ("Пароль из соли и перца.", "not_food", "hard:metaphor:пароль"),
    ("Рецепт миграции базы данных.", "not_food", "hard:metaphor:рецепт"),
    ("Приготовил сборку за ночь.", "not_food", "hard:metaphor:приготовил"),
    ("Разогнать вкуснейший пайплайн.", "not_food", "hard:metaphor:пайплайн"),
    ("Добавил соль в конфиг, стало не сладко.", "not_food", "hard:metaphor:конфиг"),
    # ---------- отрицание (по policy метка не меняется) ----------
    ("Я не ем сахар.", "food", "hard:neg:есть"),
    ("В доме не было еды.", "food", "hard:neg:было"),
    ("Не купил хлеб в магазине.", "food", "hard:neg:куп"),
    ("Отказался от мяса.", "food", "hard:neg:отказ"),
    ("Блюдо без соли и специй.", "food", "hard:neg:без"),
    ("Не ем грибы и морепродукты.", "food", "hard:neg:не_ем"),
    ("Никакой еды в холодильнике.", "food", "hard:neg:никакой"),
    ("Мне не нравится грибной суп.", "food", "hard:neg:не_нравится"),
    # ---------- «съедобно звучит, но не еда» ----------
    ("Пробка от бутылки на столе.", "not_food", "hard:look:пробка"),
    ("Пенопластовая крошка на подоконнике.", "not_food", "hard:look:пенопласт"),
    ("Стружка в мастерской.", "not_food", "hard:look:стружка"),
    ("Опилки в мешке.", "not_food", "hard:look:опилки"),
    ("Кора дуба в саду.", "not_food", "hard:look:кора"),
    ("Песок в лотке.", "not_food", "hard:look:песок"),
    ("Земля в горшке.", "not_food", "hard:look:земля"),
    ("Болотный мох на камне.", "not_food", "hard:look:мох"),
    ("Панцирь черепахи.", "not_food", "hard:look:панцирь"),
    ("Клейстер в школьном кабинете.", "not_food", "hard:look:клейстер"),
    ("Копчёная кожа в обуви.", "not_food", "hard:look:кожа"),
    ("Мех от шубы.", "not_food", "hard:look:мех"),
    ("Шерсть на свитере.", "not_food", "hard:look:шерсть"),
    ("Перо на подушке.", "not_food", "hard:look:перо"),
    ("Найденная кость в парке.", "not_food", "hard:look:кость"),
    ("Семена подсолнечника в пакете.", "food", "hard:look:семена:food"),
    # ---------- не для человека ----------
    ("Корм для кошки в миске.", "not_food", "hard:adjacent:корм"),
    ("Кормушка с рыбьим кормом.", "not_food", "hard:adjacent:корм"),
    ("Косточка для собаки из зоомагазина.", "not_food", "hard:adjacent:косточка"),
    ("Сено для кролика.", "not_food", "hard:adjacent:сено"),
    # ---------- мета-лексика: «продукт», «сырьё», «блюдо» и т. п. ----------
    ("Продукты в кассе и список покупок.", "food", "hard:meta:продукт:food"),
    ("Какие продукты нельзя провозить в самолёте.", "food", "hard:meta:продукт:food"),
    ("Продукт питания с истёкшим сроком.", "food", "hard:meta:продукт:food"),
    ("Продукт смартона вышел из строя.", "not_food", "hard:meta:продукт:not_food"),
    ("Продукт труда преподавателя.", "not_food", "hard:meta:продукт:not_food"),
    ("Сырьё для борща.", "food", "hard:meta:сырьё:food"),
    ("Ингредиенты в рецепте на две порции.", "food", "hard:meta:ингредиент:food"),
    ("Состав блюда на этикетке.", "food", "hard:meta:состав:food"),
    ("Сырьё для завода привезли вагоном.", "not_food", "hard:meta:сырьё:not_food"),
    ("Ингредиент в составе крема для лица.", "not_food", "hard:meta:ингредиент:not_food"),
    ("Состав отряда на совещании.", "not_food", "hard:meta:состав:not_food"),
    ("Блюдо тарелки с синим узором.", "not_food", "hard:meta:блюдо:not_food"),
    ("Гарнир в юбке по пояс.", "not_food", "hard:meta:гарнир:not_food"),
    ("Перекус в парламенте затянулся.", "not_food", "hard:meta:перекус:not_food"),
    ("Буфет на верхней площадке оперы.", "not_food", "hard:meta:буфет:not_food"),
    ("Шведский буфет в отеле до шести.", "food", "hard:meta:буфет:food"),
    ("Сервировка стола заняла полчаса.", "food", "hard:meta:сервировка:food"),
    ("Добавка к смете на проект.", "not_food", "hard:meta:добавка:not_food"),
    ("Пищевая добавка Е330 в составе.", "food", "hard:meta:добавка:food"),
    ("Столовая в офисе закрыта на ремонт.", "food", "hard:meta:столовая:food"),
    ("Бизнес-ланч в столовой", "food", "hard:meta:ланч:food"),
    ("ВкусВилл доставит продукты за час.", "food", "hard:meta:бренд:food"),
    ("Яндекс Еда, минимальный заказ двести.", "food", "hard:meta:бренд:food"),
    ("Самокат привёз заказанное.", "food", "hard:meta:бренд:food"),
    ("Пицца.ру не работает в моём районе.", "food", "hard:meta:бренд:food"),
    ("Магнит продаёт бытовую технику.", "not_food", "hard:meta:бренд:not_food"),
    ("Техника в Магните дешевле.", "not_food", "hard:meta:бренд:not_food"),
    # ---------- жанровые сдвиги: тот же класс, другой регистр ----------
    ("что приготовить из курицы на ужин", "food", "hard:genre:запрос:food"),
    ("доставка еды из пиццерии рядом", "food", "hard:genre:запрос:food"),
    ("где купить свежую рыбу", "food", "hard:genre:запрос:food"),
    ("сколько стоит яблочный сок в литре", "food", "hard:genre:запрос:food"),
    ("ужин готов, все за столом", "food", "hard:genre:высказывание:food"),
    ("продам самогодный торт на заказ", "food", "hard:genre:объявление:food"),
    ("объявление: продаю вишнёвое варенье, банки по два литра", "food", "hard:genre:объявление:food"),
    ("купить дрель с набором бит", "not_food", "hard:genre:запрос:not_food"),
    ("где купить стул в офис", "not_food", "hard:genre:запрос:not_food"),
    ("цена на холодильник в марте", "not_food", "hard:genre:запрос:not_food"),
    ("всё собрал, наконец-то", "not_food", "hard:genre:высказывание:not_food"),
    ("объявление: продам велосипед, недорого", "not_food", "hard:genre:объявление:not_food"),
    ("сдам в аренду гараж на две машины", "not_food", "hard:genre:объявление:not_food"),
    # ---------- короткие / без пунктуации / keywords-only ----------
    ("яблочный пирог!", "food", "hard:genre:шум:food"),
    ("нож", "not_food", "hard:genre:шум:not_food"),
    ("пирог яблоко начинка", "food", "hard:genre:ключевые:food"),
    ("холодильник сломался", "not_food", "hard:genre:ключевые:not_food"),
]

# Голые эмодзи сюда НЕ попадают осознанно: после токенизации в них не остаётся ни одного
# значимого токена, и метка становится произвольной. Эмодзи ниже добавляются только
# как хвост к обычному тексту — в SURFACE_TAILS.


def build_hard_pack() -> list[dict]:
    rows = []
    for text, label, group in HARD_PACK:
        if label not in VALID_LABELS:
            raise ValueError(f"неизвестная метка {label!r} в строке: {text}")
        if not text.strip():
            raise ValueError(f"пустой текст в группе {group}")
        rows.append({
            TEXT_COL: text.strip(),
            "label": label,
            "label_id": LABEL2ID[label],
            "group": group,
            "difficulty": "hard",
            "source": "hard_pack",
            "frame": "hand_written",
            "entity": group.split(":")[2] if len(group.split(":")) > 2 else group,
        })
    return rows


hard_rows = build_hard_pack()
_hard_df = pd.DataFrame(hard_rows)
print(f"хард-строк: {len(hard_rows)}, групп: {_hard_df['group'].nunique()}")
print(_hard_df['label'].value_counts().to_string())
_hard_df.head(8)

## 5. Генерация из лексики

Одна сущность → `FRAMES_PER_ENTITY` фреймов → каждый текст иногда получает одну surface-пертурбацию.

Пертурбации безопасны по построению: у каждой есть **якорь** (слово сущности) и проверка —
после изменения текст обязан сохранить якорь. Иначе пертурбация, которая стёрла бы
существительное, тихо сломала бы метку.

In [ ]:
SURFACE_PREFIXES = ["", "", "", "ну ", "а ", "кстати, ", "...", "ой, "]
SURFACE_TAILS = ["", "", "", " 🤔", " (фото)", " (с моего телефона)", " пжл", " напишите",
                 " подскажите", " 😅"]
STOPWORDS = {"на", "в", "с", "и", "не", "а", "из", "по", "для", "у", "о", "к", "за", "то", "же", "бы"}


def cap(text: str) -> str:
    return text[0].upper() + text[1:] if text else text


def _op_lower_first(text, rng):
    return text[0].lower() + text[1:]


def _op_upper(text, rng):
    return text.upper()


def _op_drop_punct(text, rng):
    return text.rstrip(".!?")


def _op_punct(text, rng):
    return text.rstrip(".!?") + rng.choice(["!", "?", "...", "!!", "?"])


def _op_prefix(text, rng):
    p = rng.choice(SURFACE_PREFIXES)
    return (p + text[0].lower() + text[1:]) if p and text else text


def _op_tail(text, rng):
    t = rng.choice(SURFACE_TAILS)
    return text + t if t else text


def _op_typo(text, rng):
    words = [w for w in re.findall(r"[А-Яа-яЁёA-Za-z]{5,}", text) if w.lower() not in STOPWORDS]
    if not words:
        return text
    w = rng.choice(words)
    i = rng.randrange(1, len(w) - 1)
    return text.replace(w, w[:i] + w[i] + w[i:], 1)


def _op_keywords(text, rng):
    tokens = re.findall(r"[\w-]+", text)
    kept = [t for t in tokens if t.lower() not in STOPWORDS]
    return " ".join(kept) if kept else text


def _op_spaces(text, rng):
    parts = text.split(" ")
    if len(parts) < 3:
        return text
    i = rng.randrange(1, len(parts))
    return " ".join(parts[:i] + [""] + parts[i:])


SURFACE_OPS = [
    _op_lower_first, _op_upper, _op_drop_punct, _op_punct, _op_prefix,
    _op_tail, _op_typo, _op_keywords, _op_spaces,
]


def perturb(text: str, anchor: str, rng: random.Random) -> str:
    """Одна случайная пертурбация. Возвращает исходный текст, если результат
    потерял якорь (слово сущности) или превратился в мусор."""
    for _ in range(6):
        out = rng.choice(SURFACE_OPS)(text, rng)
        out = re.sub(r"\s+", " ", out).strip()
        if not out:
            continue
        if anchor.lower() not in out.lower():
            continue
        if not re.search(r"[а-яёa-z]{3,}", out):
            continue
        return out
    return text


def render_frame(frame: str, entity: dict, others: list[dict], label: str, rng: random.Random) -> str:
    o2, o3 = (rng.choice(others), rng.choice(others))
    ctx = {
        "nom": entity["nom"], "Nom": cap(entity["nom"]),
        "gen": entity["gen"], "pl": entity["pl"],
        "adj": rng.choice(ADJ[label][entity["gender"]]),
        "qual": rng.choice(QUAL[label][entity["gender"]]),
        "surf": rng.choice(SURF),
        "room": rng.choice(ROOMS),
        "meal": rng.choice(MEALS),
        "act": rng.choice(FOOD_ACTS if label == "food" else NF_ACTS),
        "nom2": o2["nom"], "nom3": o3["nom"],
    }
    return re.sub(r"\s+", " ", frame.format(**ctx)).strip()


def generate_from_lexicon(entities: list[dict], rng: random.Random) -> list[dict]:
    by_label = {lbl: [e for e in entities if e["label"] == lbl] for lbl in VALID_LABELS}
    rows = []
    for entity in sorted(entities, key=lambda e: e["group"]):
        label = entity["label"]
        frames = rng.sample(FRAME_SETS[label], k=min(FRAMES_PER_ENTITY, len(FRAME_SETS[label])))
        for frame in frames:
            text = render_frame(frame, entity, by_label[label], label, rng)
            if not text:
                continue
            variants = [text]
            if rng.random() < PERTURB_PROB:
                variants.append(perturb(text, entity["nom"], rng))
            for variant in dict.fromkeys(variants):
                rows.append({
                    TEXT_COL: variant,
                    "label": label,
                    "label_id": LABEL2ID[label],
                    "group": entity["group"],
                    "difficulty": "easy",
                    "source": "lexicon_gen",
                    "frame": frame,
                    "entity": entity["nom"],
                })
    return rows


rng = random.Random(SEED)
gen_rows = generate_from_lexicon(ENTITIES, rng)
print(f"сгенерировано строк: {len(gen_rows)}")
print(pd.Series([r["label"] for r in gen_rows]).value_counts().to_string())
print(pd.Series([r["frame"] for r in gen_rows]).nunique(), "разных фреймов использовано")

## 6. Seed из v1

Те самые 250 подписей. Сохраняем — но **группируем по голове существительного**, иначе
`Картофель, отварной` и `Картофель фри` разъедутся по сплитам и снова дадут утечку.
Если голова совпадает с сущностью из лексики, берётся её группа, иначе — своя `seed:*`.

In [ ]:
SEED_PATH = DATA_DIR / "seed_captions.json"
seed_json = json.loads(SEED_PATH.read_text(encoding="utf-8"))

_entity_by_word = {e["first_word"]: e for e in ENTITIES}


def seed_group(caption: str) -> str:
    head = re.findall(r"[а-яёa-z]+", caption.lower())[0]
    if head in _entity_by_word:
        return _entity_by_word[head]["group"]
    for e in ENTITIES:                      # запасной вариант: по любому слову
        if head in (e["first_word"], e["gen"].split()[0], e["nom"].split()[-1]):
            return e["group"]
    return f"seed:{head}"


seed_rows = []
for label, captions in seed_json.items():
    for caption in captions:
        head = re.findall(r"[а-яёa-z]+", caption.lower())[0]
        seed_rows.append({
            TEXT_COL: caption,
            "label": label,
            "label_id": LABEL2ID[label],
            "group": seed_group(caption),
            "difficulty": "easy",
            "source": "manual_seed",
            "frame": "v1_template",
            "entity": head,
        })

# часть seedов тоже получает surface-вариант, иначе 250 шаблонных строк
# останутся единственными представителями этого регистра
seed_rng = random.Random(SEED + 1)
expanded_seed_rows = []
for row in seed_rows:
    expanded_seed_rows.append(row)
    if seed_rng.random() < PERTURB_PROB:
        variant = perturb(row[TEXT_COL], row["entity"], seed_rng)
        if variant != row[TEXT_COL]:
            new_row = dict(row, **{TEXT_COL: variant})
            expanded_seed_rows.append(new_row)

_seed_groups = {r["group"] for r in seed_rows}
_matched_groups = [g for g in _seed_groups if not g.startswith("seed:")]
print(f"seed-строк: {len(seed_rows)} -> с вариантами: {len(expanded_seed_rows)}")
print(f"seed-групп: {len(_seed_groups)}")
print(f"из них уехали в существующие группы: {len(_matched_groups)}")

## 7. Сборка

Контроль прямо здесь: одинаковый текст с двумя разными метками — это ошибка разметки,
а не повод «оставить как есть». Такие строки падают в assert и чинятся руками.

In [ ]:
COLUMNS = [TEXT_COL, "label", "label_id", "group", "difficulty", "source", "frame", "entity"]

df = pd.DataFrame(gen_rows + hard_rows + expanded_seed_rows, columns=COLUMNS)

# 1) один текст — одна метка
_by_text = df.groupby(TEXT_COL)["label"].nunique()
_conflicts = _by_text[_by_text > 1]
assert len(_conflicts) == 0, (
    f"один и тот же текст с разными метками ({len(_conflicts)} шт.): "
    f"{list(_conflicts.index[:5])}"
)

# 2) дубликаты (в т.ч. между разными источниками) — оставляем первый
n_before = len(df)
df = df.drop_duplicates(subset=[TEXT_COL], keep="first").reset_index(drop=True)
print(f"дубликатов удалено: {n_before - len(df)}")

# 3) перемешиваем детерминированно
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"\nвсего строк: {len(df)}")
print(df["label"].value_counts().to_string())
print("\nпо источникам:")
print(df.groupby(["source", "label"]).size().unstack(fill_value=0))
print("\nпо сложности:")
print(df.groupby(["difficulty", "label"]).size().unstack(fill_value=0))
print(f"\nуникальных текстов: {df[TEXT_COL].nunique()}, групп: {df['group'].nunique()}")
df.head(10)

## 8. Entity-level split

Сплит делается **по группам**, а не по строкам: одна группа = одна сущность, и она целиком
уходит в один split. Иначе `Картофель, отварной` в train и `Картофель фри` в test дают
100% на тесте, не давая ничего.

Разделы разные:

* обычные группы — `70 / 15 / 15` (train / validation / test);
* хард-группы — `55 / 15 / 30`, и хвост уходит в **`hard_test`**, а не в `test`.

`hard_test` — это 100% хард-примеров с группами, которых не было в train. На нём
и смотрят, выучила ли модель смысл или словарь.

In [ ]:
# Доли: обычные группы 70/15/15, хард-группы 55/15/30 с хвостом в hard_test.
SPLIT_FRACTIONS = {
    "easy": [("train", 0.70), ("validation", 0.15), ("test", 0.15)],
    "hard": [("train", 0.55), ("validation", 0.15), ("hard_test", 0.30)],
}


def assign_splits(frame: pd.DataFrame) -> pd.Series:
    """Раскладка по группам, случайная и сбалансированная по меткам.

    Раньше группы сортировались по имени и резались по индексу, а имена начинаются
    с food: / hard: / nf: / seed:. Из-за этого train забирал почти всю еду, а
    validation и test уезжали в not_food — сплит получался не случайным, а по
    префиксу. Теперь группы перемешиваются (SEED) и раскладываются жадно в сплит
    с наибольшей относительной недоборкой, а еда и не-еда в очереди чередуются.
    """
    stats = frame.groupby("group").agg(
        n=("label", "size"),
        hard=("difficulty", lambda s: bool((s == "hard").all())),
        food=("label", lambda s: float((s == "food").mean())),
    )

    rng = random.Random(SEED)
    mapping = {}

    for bucket, fractions in SPLIT_FRACTIONS.items():
        part = stats[stats["hard"] == (bucket == "hard")]

        pools = {}
        for is_food in (True, False):
            names = sorted(part.index[(part["food"] >= 0.5) == is_food])
            rng.shuffle(names)
            pools[is_food] = names

        queue = []
        for i in range(max(len(pools[True]), len(pools[False]))):
            for pool in (pools[True], pools[False]):
                if i < len(pool):
                    queue.append(pool[i])

        targets = {s: frac * part["n"].sum() for s, frac in fractions}
        filled = dict.fromkeys(targets, 0.0)
        for g in queue:
            split = max(targets, key=lambda s: (targets[s] - filled[s]) / targets[s])
            mapping[g] = split
            filled[split] += part.loc[g, "n"]

    return frame["group"].map(mapping)


df["split"] = assign_splits(df)
df = df.sort_values("split").reset_index(drop=True)

SPLIT_ORDER = ["train", "validation", "test", "hard_test"]
_counts = df.groupby(["split", "label"]).size().unstack(fill_value=0).reindex(SPLIT_ORDER)
_counts["share_food"] = (_counts.get("food", 0) / _counts.sum(axis=1)).round(3)
_counts["groups"] = df.groupby("split")["group"].nunique().reindex(SPLIT_ORDER)
_counts
print("hard_test состоит только из хард-примеров:",
      bool((df.loc[df["split"] == "hard_test", "difficulty"] == "hard").all()))

## 9. Валидация датасета

Про датасет, а не про модель. Каждая проверка ловит ровно тот способ, которым v1
набирал 100%.

Ключевая метрика качества датасета — **не** accuracy, а разрыв между простым
keyword-baseline и реальной моделью:

* если keyword-baseline (словарь из train) даёт 100% на `test` — датасет тривиален,
  учить бессмысленно, чинить надо данные;
* если он даёт 100% и на `hard_test` — хард-пакет не работает;
* если baseline проваливается на `hard_test`, а модель справляется — модель выучила смысл.

In [ ]:
import itertools

report_rows = []


def check(name, ok, detail=""):
    report_rows.append({"проверка": name, "ok": bool(ok), "деталь": detail})


# --- 1. структура сплитов ---
check("все сплиты непустые", set(df["split"].unique()) == set(SPLIT_ORDER),
      str(df["split"].value_counts().to_dict()))
for split in SPLIT_ORDER:
    labels = set(df.loc[df["split"] == split, "label"])
    check(f"в {split} есть обе метки", labels == VALID_LABELS, str(sorted(labels)))
for split in SPLIT_ORDER:
    share = df.loc[df["split"] == split, "label"].eq("food").mean()
    check(f"{split}: доля food в [0.35, 0.65]", 0.35 <= share <= 0.65, f"{share:.3f}")

# --- 2. утечки ---
_group_splits = df.groupby("group")["split"].nunique()
_leak_groups = sorted(_group_splits[_group_splits > 1].index)
check("утечки по группам нет", not _leak_groups, f"{len(_leak_groups)} групп: {_leak_groups[:5]}")

_text_splits = df.groupby(TEXT_COL)["split"].nunique()
_leak_texts = sorted(_text_splits[_text_splits > 1].index)
check("утечки по тексту нет", not _leak_texts, f"{len(_leak_texts)}: {_leak_texts[:5]}")

# --- 3. фрейм не выдаёт метку ---
_frame_purity = df.groupby("frame")["label_id"].mean()
_shared_frame_ids = set(SHARED_FRAMES)
_dirty = {f: round(float(_frame_purity[f]), 3) for f in _shared_frame_ids
          if pd.notna(_frame_purity.get(f, np.nan))
          and not 0.1 <= float(_frame_purity[f]) <= 0.9}
check("общие фреймы не выдают метку", not _dirty, f"грязные: {list(_dirty.items())[:5]}")

# --- 4. длина не выдаёт метку ---
_len = df.assign(n=df[TEXT_COL].str.split().str.len()).groupby("label")["n"].mean()
_len_ratio = float(max(_len) / min(_len))
check("длина не выдаёт метку (разница средних < 15%)", _len_ratio < 1.15,
      f"средняя длина: {dict(_len.round(2))}")

# --- 5. keyword-baseline: словарь из train ---
def _wordset(s: str) -> set:
    return set(re.findall(r"[а-яёa-z]{3,}", s.lower()))


_train = df[df["split"] == "train"]
_food_words = {w for s in _train.loc[_train["label"] == "food", TEXT_COL] for w in _wordset(s)}
_nf_words = {w for s in _train.loc[_train["label"] == "not_food", TEXT_COL] for w in _wordset(s)}


def keyword_predict(text: str) -> str:
    words = _wordset(text)
    f, n = len(words & _food_words), len(words & _nf_words)
    if f == n:
        return "not_food"
    return "food" if f > n else "not_food"


for split in ["test", "hard_test"]:
    sub = df[df["split"] == split]
    acc = float((sub[TEXT_COL].map(keyword_predict) == sub["label"]).mean())
    check(f"keyword-baseline на {split}", True, f"accuracy = {acc:.3f} ({len(sub)} примеров)")

REPORT = pd.DataFrame(report_rows)
REPORT

In [ ]:
# --- 6. TF-IDF baseline по группам: сколько в датасете вообще сигнала ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(
    TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_df=0.9),
    LogisticRegression(max_iter=2000, C=2.0),
)

gkf = GroupKFold(n_splits=5)
pred = np.zeros(len(df), dtype=int)
for tr, te in gkf.split(df, df["label_id"], groups=df["group"]):
    pipe.fit(df.iloc[tr][TEXT_COL], df.iloc[tr]["label_id"])
    pred[te] = pipe.predict(df.iloc[te][TEXT_COL])

df["tfidf_pred"] = pred
_is_hard = df["difficulty"].eq("hard").to_numpy()
_acc_all = float((pred == df["label_id"].to_numpy()).mean())
_acc_hard = float((pred[ _is_hard] == df["label_id"].to_numpy()[ _is_hard]).mean())
_acc_easy = float((pred[~_is_hard] == df["label_id"].to_numpy()[~_is_hard]).mean())

print(f"TF-IDF + LogReg, grouped 5-fold:  overall { _acc_all:.3f}   "
      f"easy { _acc_easy:.3f}   hard { _acc_hard:.3f}")
print("\nесли hard ≈ easy, значит хард-пакет слишком лёгкий и учит ровно тому же, что и основная часть\n")

print("10 ошибок TF-IDF на хард-примерах (посмотреть глазами):")
bad = df[(_is_hard) & (pred != df["label_id"].to_numpy())]
display(bad[[TEXT_COL, "label", "group"]].head(10))

REPORT.loc[len(REPORT)] = {
    "проверка": "TF-IDF baseline (grouped CV)",
    "ok": True,
    "деталь": f"overall {_acc_all:.3f}, easy {_acc_easy:.3f}, hard {_acc_hard:.3f}, "
              f"ошибок на hard: {int((pred[_is_hard] != df['label_id'].to_numpy()[_is_hard]).sum())}",
}
REPORT

## 10. Публикация

* на диск — `data/food_not_food_v2/{split}.jsonl` + parquet;
* в HuggingFace — `DatasetDict` с `ClassLabel` (id зашиты и больше не зависят от порядка строк);
* `dataset_card.md` — чтобы правило разметки жило вместе с данными;
* `PUSH_TO_HUB = True` перезаливает репозиторий.

In [ ]:
from datasets import Dataset

KEEP = [TEXT_COL, "label", "label_id", "group", "difficulty", "source", "frame", "entity"]
out = df[KEEP + ["split"]].copy()          # caption, label, ... — как в v1

# Схема как в v1: {"caption": ..., "label": "food"} — метка строковая, и caption и
# label идут первыми колонками. Фиксированные id живут в label_id, границы сплита —
# в колонке split (сплит сделан по группам, см. assign_splits), так что train/test
# вырезается самому одной строкой.
_unknown = sorted(set(out["label"]) - set(LABEL2ID))
assert not _unknown, f"метки вне LABEL2ID: {_unknown}"
assert (out["label"].map(LABEL2ID).to_numpy() == out["label_id"].to_numpy()).all(), \
    "label и label_id разошлись"
assert list(out.columns[:2]) == [TEXT_COL, "label"], list(out.columns[:2])

ds = Dataset.from_pandas(out)

with (OUT_DIR / "all.jsonl").open("w", encoding="utf-8") as f:
    for row in out.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
for split in SPLIT_ORDER:          # те же строки, нарезанные по сплитам
    with (OUT_DIR / f"{split}.jsonl").open("w", encoding="utf-8") as f:
        for row in out.loc[out["split"] == split, KEEP].to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

out.to_parquet(OUT_DIR / "food_not_food_v2.parquet", index=False)
print(ds)

print("\nпервая строка:", out.iloc[0].to_dict())
print("\nразмеры по сплитам:")
print(out.groupby(["split", "label"]).size().unstack(fill_value=0).reindex(SPLIT_ORDER))

In [ ]:
_size_table = "\n".join(
    [f"* всего: {len(out)} строк, {out['group'].nunique()} групп"]
    + [f"* `{s}`: {(out['split'] == s).sum()} строк, "
       f"{out.loc[out['split'] == s, 'group'].nunique()} групп" for s in SPLIT_ORDER]
)

card = f"""---
license: mit
task_categories:
  - text-classification
language: ru
---

# food / not_food (v{LABEL_POLICY['version']})

Двоичная классификация русскоязычного текста: относится ли он к съедобным для человека
продуктам или блюдам.

Строки выглядят как в v1 — `{{"caption": ..., "label": "food"}}`. `label` строковый,
чтобы работал прежний пайплайн; фиксированные id — в `label_id` (`not_food`=0,
`food`=1), границы сплита — в колонке `split`.

Схема на диске (`*.jsonl`, `food_not_food_v2.parquet`) и на HuggingFace
одинаковая: `label` везде int, `ClassLabel` c именами `not_food` / `food`.

## Правило разметки

{LABEL_POLICY['rule']}

* {LABEL_POLICY['sense_over_string']}
* {LABEL_POLICY['negation']}
* {LABEL_POLICY['scope_note']}

## Схема

| колонка | тип | смысл |
|---|---|---|
| `caption` | string | текст |
| `label` | string | `food` / `not_food`, как в v1 |
| `split` | string | `train` / `validation` / `test` / `hard_test`, границы по группам |
| `label_id` | int64 | фиксированный id: `not_food`=0, `food`=1 |
| `group` | string | сущность; сплит сделан по группам |
| `difficulty` | string | `easy` / `hard` |
| `source` | string | `lexicon_gen` / `hard_pack` / `manual_seed` |
| `frame` | string | использованный фрейм |
| `entity` | string | опорная сущность |

## Сплиты

Весь датасет лежит одной таблицей с колонкой `split`, так что его можно и резать самому.
Готовые границы: `train` / `validation` / `test` — обычные примеры, сплит по группам
(одна сущность целиком в одном сплите, 70/15/15); `hard_test` — только хард-примеры,
сплит 55/15/30. Ни одна группа не встречается в двух сплитах.

## Размер

{_size_table}

## Ограничения

* только русский язык; латиница, code-switching и английский текст не представлены;
* баланс классов 50/50 — в реальном потоке `food` встречается заметно реже,
  поэтому модель будет переоценивать `food`; лечится калибровкой порога;
* хард-пакет — ручная разметка, он отражает решение о краевых случаях из LABEL_POLICY,
  а не универсальную истину; при изменении правила его надо пересмотреть.
"""

(OUT_DIR / "dataset_card.md").write_text(card, encoding="utf-8")
print(card)
print(f"записано в {OUT_DIR.resolve()}")

In [ ]:
def _front_matter(text: str) -> tuple[dict, str]:
    """Отрезает YAML-шапку ноутбучной карточки, возвращает (метаданные, тело)."""
    parts = text.split("---", 2)
    if len(parts) != 3:
        return {}, text
    import yaml
    return yaml.safe_load(parts[1]) or {}, parts[2]


if PUSH_TO_HUB:
    ds.push_to_hub(HF_REPO_ID, split="train")

    # push_to_hub генерирует README с пустым описанием, поэтому мержим нашу карточку
    # (правило разметки, сплиты, ограничения) с его авто-YAML: иначе на hub нет
    # даже описания задачи.
    import yaml
    from huggingface_hub import hf_hub_download, upload_file

    _card_meta, _card_body = _front_matter((OUT_DIR / "dataset_card.md").read_text(encoding="utf-8"))
    _auto_readme = Path(hf_hub_download(HF_REPO_ID, "README.md", repo_type="dataset"))
    _auto_meta, _ = _front_matter(_auto_readme.read_text(encoding="utf-8"))

    _meta = {**_auto_meta, **_card_meta}          # наша карточка главнее
    if isinstance(_meta.get("task_categories"), str):   # hub требует список, а не строку
        _meta["task_categories"] = [_meta["task_categories"]]
    # закрывающий --- обязан быть на отдельной строке, иначе hub считает карточку
    # пустой и теряет dataset_info, license и language
    _readme = ("---\n" + yaml.safe_dump(_meta, allow_unicode=True, sort_keys=False)
               + "---\n" + _card_body.lstrip("\n"))
    upload_file(path_or_fileobj=_readme.encode("utf-8"), path_in_repo="README.md",
                repo_id=HF_REPO_ID, repo_type="dataset")
    print(f"загружено в {HF_REPO_ID} (данные + карточка)")
else:
    print(f"PUSH_TO_HUB = False — локально всё лежит в {OUT_DIR.resolve()}.\n"
          f"Поставить PUSH_TO_HUB = True, чтобы перезалить {HF_REPO_ID}.")

print("следующий шаг: в text_classification.ipynb взять этот датасет вместо v1, "
      "зашить id2label константой и мерить метрику отдельно на test и hard_test.")
df.groupby(["split", "source"]).size().unstack(fill_value=0).reindex(SPLIT_ORDER)